In [1]:
import os
from pathlib import Path
import cv2
from pyzbar.pyzbar import decode
import pandas as pd

def scan_qrs_from_target_images():
    download_dir = Path.home() / "Downloads"
    
    target_filenames = [
        "S__83165253_0",
        "S__83165254_0",
        "S__83165255_0",
        "S__83165256_0"
    ]
    possible_extensions = [".jpg", ".jpeg", ".png", ".webp"]
    
    extracted_data = []

    print(f"🔍 Starting QR Code Extraction from Downloads...\n")

    for name in target_filenames:
        image_path = None
        # 1. ค้นหาไฟล์ตามนามสกุล
        for ext in possible_extensions:
            p = download_dir / f"{name}{ext}"
            if p.exists():
                image_path = p
                break
                
        if not image_path:
            print(f"❌ ไม่พบไฟล์: {name}")
            continue

        # 2. อ่านไฟล์รูปภาพ
        img = cv2.imread(str(image_path))
        if img is None:
            print(f"⚠️ ไม่สามารถเปิดไฟล์ภาพได้: {image_path.name}")
            continue

        # 3. Decode หา QR Code ทั้งหมดในภาพ
        qrs = decode(img)
        print(f"🖼️  {image_path.name}: พบทั้งหมด {len(qrs)} QR Code")

        # 4. จัดเรียงลำดับ QR Code ตามตำแหน่งในภาพ (บนลงล่าง, ซ้ายไปขวา)
        # ป้องกันไม่ให้ลำดับของแผ่นที่มีหลาย QR (เช่น 16 ตัว) สลับกัน
        qrs_sorted = sorted(qrs, key=lambda q: (q.rect.top, q.rect.left))

        for idx, qr in enumerate(qrs_sorted, 1):
            try:
                # แปลง Byte data เป็น string URL
                link = qr.data.decode('utf-8')
                
                extracted_data.append({
                    "Source_File": image_path.name,
                    "QR_Index": idx,
                    "Link": link,
                    "Position_Y": qr.rect.top,
                    "Position_X": qr.rect.left
                })
            except Exception as e:
                print(f"   ⚠️ ไม่สามารถดึงข้อมูลจาก QR #{idx} ได้: {e}")

    return pd.DataFrame(extracted_data)

# ==========================================
# 🚀 Run Extraction & Save Results
# ==========================================
if __name__ == "__main__":
    df_links = scan_qrs_from_target_images()

    print("\n" + "="*60)
    print("📌 ผลลัพธ์การสกัด ลิงก์จาก QR Code (Sample):")
    print("="*60)
    
    if not df_links.empty:
        # แสดงผลลัพธ์บางส่วน
        print(df_links[["Source_File", "QR_Index", "Link"]].to_string(index=False))

        # ส่งออกไฟล์ CSV สำหรับนำไปสร้าง Portal ต่อได้ทันที
        output_csv = Path.home() / "Downloads" / "extracted_stroke_links.csv"
        df_links.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"\nบันทึกไฟล์ผลลัพธ์ไว้ที่: {output_csv}")
    else:
        print("ไม่พบลิงก์จาก QR Code ในภาพที่ระบุ")

🔍 Starting QR Code Extraction from Downloads...

🖼️  S__83165253_0.jpg: พบทั้งหมด 1 QR Code
🖼️  S__83165254_0.jpg: พบทั้งหมด 15 QR Code
🖼️  S__83165255_0.jpg: พบทั้งหมด 1 QR Code
🖼️  S__83165256_0.jpg: พบทั้งหมด 1 QR Code

📌 ผลลัพธ์การสกัด ลิงก์จาก QR Code (Sample):
      Source_File  QR_Index                                                                                     Link
S__83165253_0.jpg         1                               https://chulabhornchannel.cra.ac.th/health-articles/23771/
S__83165254_0.jpg         1 https://youtube.com/playlist?list=PLVGq9IQHt11YfUPZPPnnnvq4KgaWRY3ZF&si=MS6MoIyjYTQzAGTG
S__83165254_0.jpg         2 https://youtube.com/playlist?list=PLVGq9IQHt11Zvao7gpzcZkFKVhLFSNgaT&si=6iUzwzqeLeh9v6j0
S__83165254_0.jpg         3    https://drive.google.com/file/d/1JdzU-8qLQNBqfkcLclKQp5OOHJpjqp8F/view?usp=drive_link
S__83165254_0.jpg         4 https://youtube.com/playlist?list=PLVGq9IQHt11bLanp3R3L8xys9iCyCzH2K&si=elzFhpAHBPe4DRiU
S__83165254_0.jpg         5 htt

In [4]:
import os
import re
from pathlib import Path
import cv2
import requests
from bs4 import BeautifulSoup
from pyzbar.pyzbar import decode
import pandas as pd

# Header หลอกเซิร์ฟเวอร์เพื่อให้เข้าอ่านเว็บได้ไม่โดนบล็อก
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def fetch_web_title(url):
    """ยิง Request เข้าไปอ่าน <title> หรือ og:title ของเว็บ/คลิป YouTube/Google Form"""
    try:
        # กรณีเป็น short url ของ Google Forms หรือ YouTube ให้ดึง URL จริงก่อน (Allow redirects)
        response = requests.get(url, headers=HEADERS, timeout=6, allow_redirects=True)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 1. ลองหา og:title ก่อน (มักจะมีชื่อเรื่องภาษาไทยสวยๆ ใน YouTube/Facebook/Web)
            og_title = soup.find("meta", property="og:title")
            if og_title and og_title.get("content"):
                return og_title["content"].strip()
                
            # 2. ถ้าไม่มี og:title ให้เอา <title> ปกติ
            if soup.title and soup.title.string:
                title = soup.title.string.strip()
                # ลบคำลงท้ายสเกลใหญ่ เช่น "- YouTube" หรือ "Google Forms"
                title = re.sub(r' - YouTube$| - Google Forms$', '', title)
                return title
    except Exception as e:
        pass
    return "ไม่สามารถดึงชื่อหัวข้ออัตโนมัติได้"

def preprocess_and_decode(image):
    """ปรับความคมชัดภาพหลายๆ แบบเพื่อสกัด QR Code ที่จาง/เบลอ ให้ได้ครบถ้วนที่สุด"""
    qrs = decode(image)
    if len(qrs) > 0:
        return qrs
        
    # หากยังสแกนไม่ติด ให้ลองทำ Grayscale + Thresholding
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # แบบที่ 1: Adaptive Thresholding
    thresh1 = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    qrs = decode(thresh1)
    if len(qrs) > 0:
        return qrs
        
    # แบบที่ 2: Resize ภาพให้ใหญ่ขึ้น 2 เท่า (กรณี QR ขนาดเล็กมาก)
    resized = cv2.resize(image, (0,0), fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    qrs = decode(resized)
    return qrs

def extract_and_enrich_stroke_data():
    download_dir = Path.home() / "Downloads"
    target_filenames = ["S__83165253_0", "S__83165254_0", "S__83165255_0", "S__83165256_0"]
    possible_extensions = [".jpg", ".jpeg", ".png", ".webp"]
    
    extracted_records = []
    print("🚀 เริ่มต้นกระบวนการสกัด QR Code และดึงข้อมูล Title จากหน้าเว็บจริง...\n")

    for name in target_filenames:
        image_path = None
        for ext in possible_extensions:
            p = download_dir / f"{name}{ext}"
            if p.exists():
                image_path = p
                break
                
        if not image_path:
            print(f"❌ ไม่พบไฟล์: {name}")
            continue

        img = cv2.imread(str(image_path))
        if img is None:
            continue

        # ดึง QR Code ทั้งหมด
        qrs = preprocess_and_decode(img)
        print(f"🖼️ {image_path.name}: พบทั้งหมด {len(qrs)} QR Code")

        # เรียงลำดับ Grid: จากบนลงล่าง, ซ้ายไปขวา
        qrs_sorted = sorted(qrs, key=lambda q: (q.rect.top, q.rect.left))

        for idx, qr in enumerate(qrs_sorted, 1):
            try:
                url = qr.data.decode('utf-8')
                
                print(f"   ⏳ [{idx}/{len(qrs_sorted)}] กำลังดึง Title จาก: {url[:45]}...")
                actual_title = fetch_web_title(url)
                
                # กำหนด Category เบื้องต้นจากแผ่นภาพ
                category = "ความรู้ทั่วไป"
                if "83165254" in name:
                    category = "การดูแลผู้ป่วยที่บ้าน"
                elif "83165255" in name:
                    category = "แบบประเมินผู้ป่วย"
                elif "83165256" in name:
                    category = "กายภาพบำบัด"

                extracted_records.append({
                    "Source_Image": image_path.name,
                    "QR_Position_Index": idx,
                    "Category": category,
                    "Fetched_Title": actual_title,
                    "Link": url
                })
            except Exception as e:
                print(f"   ⚠️ Error processing QR #{idx}: {e}")

    # แปลงเป็น DataFrame
    df = pd.DataFrame(extracted_records)
    
    # บันทึกเป็น Excel
    output_excel = download_dir / "stroke_links_enriched.xlsx"
    df.to_excel(output_excel, index=False)
    print(f"\n✅ ดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ!")
    print(f"📁 บันทึกไฟล์ Excel พร้อมใช้สร้างเว็บไว้ที่: {output_excel}")
    
    return df

if __name__ == "__main__":
    df_result = extract_and_enrich_stroke_data()

🚀 เริ่มต้นกระบวนการสกัด QR Code และดึงข้อมูล Title จากหน้าเว็บจริง...

🖼️ S__83165253_0.jpg: พบทั้งหมด 1 QR Code
   ⏳ [1/1] กำลังดึง Title จาก: https://chulabhornchannel.cra.ac.th/health-ar...
🖼️ S__83165254_0.jpg: พบทั้งหมด 15 QR Code
   ⏳ [1/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [2/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [3/15] กำลังดึง Title จาก: https://drive.google.com/file/d/1JdzU-8qLQNBq...
   ⏳ [4/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [5/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [6/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [7/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [8/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [9/15] กำลังดึง Title จาก: https://youtube.com/playlist?list=PLVGq9IQHt1...
   ⏳ [10/15] กำลังดึง Title จาก: ht